# ⭐ Project · The Nimbus Briefing Assistant

> **Kernel:** `Phase 5 · LangGraph` · **Needs:** Ollama and this section's
> `mcp_server.py`. The full run makes a few dozen model calls; on a local model
> expect several minutes.

Time to use everything. **Nimbus Robotics** (the fictional company in
`data/kb/`) has a problem: its Q3 rollout of 180 delivery robots to the
customer Albers Markt is slipping. On the last steering call, Albers Markt asked
for *"a written briefing with a realistic date per store group"*.

You'll build the assistant that writes that briefing. The facts are spread
across six documents (a battery shortage, a delayed software release, a permit,
customer feedback), so no single lookup answers it. The assistant has to plan,
research in parallel, write, check its own work, and get a human's approval
before anything is saved.

**What you'll practise**

| Feature | From notebook |
|---|---|
| Tools served over MCP, run asynchronously | 11 |
| A planner with structured output, grounded by an MCP resource | 03, 11 |
| Parallel research workers with `Send`, each a tool-using agent | 04, 05 |
| Retry policy and bounded concurrency on the workers | 05, 12 |
| A writer ⇄ reviewer loop with a guard (evaluator-optimizer) | 05 |
| Human approval with `interrupt()` before the side effect | 07 |
| Durable SQLite checkpoints, crash recovery, `durability='sync'` | 08 |
| Live progress with `updates` + `custom` streams | 09 |
| Token usage tracked per node | 12 |
| A graph factory that takes the model (so it's testable) | 12 |

The same project is built in every framework section of Phase 5, so you can
compare frameworks on identical work.

## 1 · The design

```
                      ┌──────────────────── Send × N (max 3 at once, retried) ─────────────────┐
 START ──► plan ──────┼─► research(q1) ─┐                                                      │
  (questions,         ├─► research(q2) ─┼──► write ──► review ──pass──► approval ──approve──► save ──► END
   grounded by the    └─► research(qN) ─┘      ▲          │               (interrupt)│
   kb://index)          (each: an agent        └─revise───┘ fail & < 2 revisions      └─reject──► END
                         with read-only MCP tools)           fail & ≥ 2 → approval (flagged)
```

Design decisions, and why:

- **Plan first, then research in parallel.** The questions are independent, so
  `Send` runs them concurrently. The planner sees the document index (an MCP
  resource), so its questions match what exists.
- **Workers get read-only tools**; only the final `save` node can write. That's
  least privilege.
- **The reviewer checks exact things in code** (a sources section, real document
  names) and asks the model only whether claims are supported by the findings.
- **The loop is capped**; a draft that still fails goes to the human
  **flagged**, never silently.
- **The side effect (saving) happens only after approval**, in its own node,
  after the `interrupt()`.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 01_LANGGRAPH folder
sys.path.insert(0, str(SECTION))

import operator, pathlib
from typing import Annotated, TypedDict, Literal
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send, Command, RetryPolicy, interrupt
from langgraph.config import get_stream_writer
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
import aiosqlite, sys

from common import get_model, SECTION_DIR

model = get_model()

## 2 · Tools and context from the MCP server

We load the server's tools once and split them by privilege. We also read the
`kb://index` resource, a list of document names and titles, which will ground
the planner.

In [ ]:
mcp = MultiServerMCPClient({'kb': {'transport': 'stdio', 'command': sys.executable,
                                   'args': [str(SECTION_DIR / 'mcp_server.py')]}})
all_tools = {t.name: t for t in await mcp.get_tools()}

READ_TOOLS = [all_tools[n] for n in ('list_documents', 'read_document', 'search_documents')]
SAVE_TOOL = all_tools['save_report']
KB_INDEX = (await mcp.get_resources('kb', uris=['kb://index']))[0].as_string()
DOC_NAMES = [line.split(':')[0] for line in KB_INDEX.splitlines()]

print('read tools :', [t.name for t in READ_TOOLS])
print('write tool :', SAVE_TOOL.name)
print(KB_INDEX)

## 3 · The state

Every key has one clear owner, or a reducer if several branches write it
(notebook 02 §8):

In [ ]:
def merge_usage(old: dict | None, new: dict) -> dict:
    out = dict(old or {})
    for node, tokens in new.items():
        out[node] = out.get(node, 0) + tokens
    return out


class Briefing(TypedDict):
    request: str                                         # input: what the customer asked for
    questions: list[str]                                 # plan
    findings: Annotated[list[str], operator.add]         # research workers (parallel) append here
    draft: str                                           # write
    feedback: str                                        # review → write
    revision: int                                        # write (loop guard)
    review_passed: bool                                  # review
    decision: str                                        # approval
    report_path: str                                     # save
    usage: Annotated[dict, merge_usage]                  # tokens per node (every LLM node)


class ResearchTask(TypedDict):                           # a research worker's private input
    request: str
    question: str


def tokens(*messages) -> int:
    return sum((m.usage_metadata or {}).get('total_tokens', 0)
               for m in messages if getattr(m, 'usage_metadata', None))

## 4 · The nodes

Each node is small and does one job. They're written inside a **factory**
(`build_briefing_graph(model, ...)`) so a test can pass a fake model (notebook 12).
`FAIL_WRITE_ONCE` lets us simulate a crash in §6.

In [ ]:
class Plan(BaseModel):
    questions: list[str] = Field(description='2 to 4 specific research questions')

class Verdict(BaseModel):
    supported: bool = Field(description='true only if every claim is backed by the findings')
    problems: str = Field(description='unsupported or wrong claims, or "none"')

MAX_REVISIONS = 2
FAIL_WRITE_ONCE = {'armed': False}          # §6 flips this to simulate a crash


def build_briefing_graph(model, read_tools, save_tool, checkpointer=None):
    planner = model.with_structured_output(Plan, method='function_calling', include_raw=True)
    checker = model.with_structured_output(Verdict, method='function_calling', include_raw=True)
    researcher = create_agent(model, read_tools, system_prompt=(
        'You research Nimbus Robotics using ONLY the knowledge-base tools. Read the relevant '
        'documents, then answer the question in at most 4 sentences with concrete numbers and dates. '
        'End with "Sources: <document names>".'))

    # ---- plan -------------------------------------------------------------
    async def plan(state: Briefing) -> dict:
        prompt = (f"Request: {state['request']}\n\nAvailable documents:\n{KB_INDEX}\n\n"
                  'Write 2-4 specific research questions whose answers are needed for this briefing.')
        out = await planner.ainvoke(prompt)
        questions = out['parsed'].questions[:4] if out['parsed'] else [state['request']]   # safe fallback
        return {'questions': questions, 'usage': {'plan': tokens(out['raw'])}}

    def fan_out(state: Briefing) -> list[Send]:
        return [Send('research', {'request': state['request'], 'question': q}) for q in state['questions']]

    # ---- research (one per question, in parallel) --------------------------
    async def research(task: ResearchTask) -> dict:
        get_stream_writer()({'researching': task['question']})
        result = await researcher.ainvoke({'messages': [
            ('user', f"Context: {task['request']}\nQuestion: {task['question']}")]})
        answer = result['messages'][-1].content
        return {'findings': [f"### {task['question']}\n{answer}"],
                'usage': {'research': tokens(*result['messages'])}}

    # ---- write --------------------------------------------------------------
    async def write(state: Briefing) -> dict:
        if FAIL_WRITE_ONCE['armed']:
            FAIL_WRITE_ONCE['armed'] = False
            raise RuntimeError('simulated crash: the writer died mid-run')
        prompt = ("Write a briefing for Albers Markt in markdown with these sections: "
                  "## Summary, ## Go-live dates per store group, ## Risks, ## Sources.\n"
                  "Use ONLY facts from the findings below. Never invent numbers or dates. "
                  "In Sources, list the knowledge-base document names you relied on.\n\n"
                  + '\n\n'.join(state['findings']))
        if state.get('feedback'):
            prompt += f"\n\nPrevious draft:\n{state['draft']}\n\nFix these problems:\n{state['feedback']}"
        reply = await model.ainvoke(prompt)
        return {'draft': reply.content, 'revision': state.get('revision', 0) + 1,
                'usage': {'write': tokens(reply)}}

    # ---- review (exact checks in code, judgement by the model) --------------
    async def review(state: Briefing) -> dict:
        draft, problems = state['draft'], []
        for section in ('## Summary', '## Go-live dates', '## Risks', '## Sources'):
            if section.lower() not in draft.lower():
                problems.append(f'missing section "{section}"')
        cited = [d for d in DOC_NAMES if d in draft]
        if not cited:
            problems.append(f'Sources must name knowledge-base documents, e.g. {DOC_NAMES[2]}')
        used = 0
        if not problems:
            out = await checker.ainvoke(
                'Findings:\n' + '\n\n'.join(state['findings']) + f'\n\nBriefing:\n{draft}\n\n'
                'Is every claim in the briefing supported by the findings? List any that are not.')
            used = tokens(out['raw'])
            if out['parsed'] and not out['parsed'].supported:
                problems.append(out['parsed'].problems)
        get_stream_writer()({'review': f"v{state['revision']}: " + ('PASS' if not problems else '; '.join(problems))[:160]})
        return {'review_passed': not problems, 'feedback': '; '.join(problems), 'usage': {'review': used}}

    def after_review(state: Briefing) -> Literal['write', 'approval']:
        if state['review_passed'] or state['revision'] >= MAX_REVISIONS:
            return 'approval'                           # flagged if it didn't pass
        return 'write'

    # ---- approval (human) ---------------------------------------------------
    def approval(state: Briefing) -> Command[Literal['save', '__end__']]:
        reply = interrupt({
            'draft': state['draft'],
            'review': 'passed' if state['review_passed']
                      else f"⚠️ FLAGGED after {state['revision']} revisions: {state['feedback']}",
        })
        if reply['action'] == 'reject':
            return Command(update={'decision': 'rejected'}, goto=END)
        draft = reply.get('text', state['draft']) if reply['action'] == 'edit' else state['draft']
        return Command(update={'decision': reply['action'], 'draft': draft}, goto='save')

    # ---- save (the only side effect, after approval) ------------------------
    async def save(state: Briefing) -> dict:
        path = await save_tool.ainvoke({'title': 'Albers Markt rollout briefing', 'markdown': state['draft']})
        return {'report_path': path if isinstance(path, str) else path[0]['text']}

    # ---- wiring ---------------------------------------------------------------
    b = StateGraph(Briefing)
    b.add_node('plan', plan)
    b.add_node('research', research,
               retry_policy=RetryPolicy(max_attempts=2, initial_interval=1.0))
    b.add_node('write', write)
    b.add_node('review', review)
    b.add_node('approval', approval)
    b.add_node('save', save)
    b.add_edge(START, 'plan')
    b.add_conditional_edges('plan', fan_out, ['research'])
    b.add_edge('research', 'write')
    b.add_edge('write', 'review')
    b.add_conditional_edges('review', after_review, ['write', 'approval'])
    b.add_edge('save', END)
    return b.compile(checkpointer=checkpointer)


print(build_briefing_graph(model, READ_TOOLS, SAVE_TOOL).get_graph().draw_ascii())

## 5 · A durable checkpointer, and a streaming runner

The graph is async (MCP tools), so it uses the **async** SQLite checkpointer.
The runner streams `updates` (which node finished) and `custom` (our progress
events), with `durability='sync'` so a crash can't lose finished research
(notebook 08), and `max_concurrency=3` so at most three workers run at once.

In [ ]:
DB = pathlib.Path('checkpoints/project.db')
DB.parent.mkdir(exist_ok=True)
for suffix in ('', '-wal', '-shm'):
    pathlib.Path(str(DB) + suffix).unlink(missing_ok=True)       # start fresh each time

conn = await aiosqlite.connect(DB)
briefing = build_briefing_graph(model, READ_TOOLS, SAVE_TOOL, checkpointer=AsyncSqliteSaver(conn))
CONFIG = {'configurable': {'thread_id': 'albers-briefing-1'}, 'max_concurrency': 3}

async def run(graph, payload, config=CONFIG):
    async for mode, data in graph.astream(payload, config, stream_mode=['updates', 'custom'],
                                          durability='sync'):
        if mode == 'custom':
            for key, value in data.items():
                print(f'   · {key}: {value}')
        else:
            for node in data:
                if node != '__interrupt__':
                    print(f'✔ {node}')
    return await graph.aget_state(config)

## 6 · Run it, with a crash in the middle

We arm `FAIL_WRITE_ONCE`, so the writer fails the first time it runs, after
planning and all research have finished and been checkpointed. Then we resume
the same thread with `None` as input. Watch which nodes run the second time.

In [ ]:
REQUEST = ('Albers Markt asked for a written briefing on the Courier C2 rollout to their 12 stores, '
           'with a realistic go-live date per store group and the main risks.')

FAIL_WRITE_ONCE['armed'] = True
try:
    await run(briefing, {'request': REQUEST, 'findings': [], 'revision': 0, 'usage': {}})
except RuntimeError as err:
    print('\n💥', err)

snapshot = await briefing.aget_state(CONFIG)
print('\ncheckpoint says next:', snapshot.next, '| findings saved:', len(snapshot.values['findings']))

In [ ]:
print('resuming the same thread...\n')
snapshot = await run(briefing, None)             # None = continue from the last checkpoint
print('\nwaiting at:', snapshot.next)

The resume started at `write`: planning and research were **not** repeated,
because their results were in the checkpoint. The run then continued through the
review loop and **paused at `approval`**, waiting for a human.

## 7 · The human reviews the briefing

In [ ]:
pending = snapshot.interrupts[0].value
print('REVIEW STATUS:', pending['review'])
print('─' * 100)
print(pending['draft'])

Read the draft critically, as the human in the loop. Are the dates consistent
with the documents? (Hint: the battery incident, the 4.2 release estimate and
the customer's "8 of 12 by 30 September" condition all matter.) Then choose:

- `{'action': 'approve'}`: save as is
- `{'action': 'edit', 'text': '...'}`: save your corrected version
- `{'action': 'reject'}`: save nothing

**What our test run showed.** The automated reviewer *passed* a draft that gave
no concrete go-live date for the Utrecht stores and listed the Utrecht permit as
a risk, although `05_permits_and_regulation` says it was approved on 8 August.
A model reviewer checks what it's asked to check, and misses what it isn't.
That's exactly why the human approval step exists, and why exercise 1 below
adds a stricter code check. Your run will differ, so read your draft the same way.

We approve here. In your own run, try `edit` and `reject` too.

In [ ]:
final = await run(briefing, Command(resume={'action': 'approve'}))
print('\ndecision   :', final.values['decision'])
print('saved to   :', final.values.get('report_path'))

## 8 · Inspect the run

The checkpoint history is a complete, replayable record of the run (notebook 08),
and the `usage` key shows where the tokens went:

In [ ]:
history = [s async for s in briefing.aget_state_history(CONFIG)]
print(f'{len(history)} checkpoints in this thread')
print('nodes that ran, in order:',
      ' → '.join(s.metadata.get('source') == 'loop' and ','.join(t.name for t in s.tasks) or '·'
                 for s in reversed(history) if s.tasks))

usage = final.values['usage']
print('\ntokens per node:', usage, '| total:', sum(usage.values()))

report = SECTION_DIR / final.values['report_path']
print(f'\nreport file: {report.relative_to(SECTION_DIR)} ({len(report.read_text())} chars)')

In [ ]:
await conn.close()          # tidy up the database connection

## 9 · Reflect: how would you improve it?

Work through these on your own. Each one exercises a concept from the section.

1. **Grounding check** (05, 12). The reviewer's model check can miss errors.
   Add a code check that every date in the draft appears in some finding.
2. **Better human review** (07). Show the human the *findings* next to the draft,
   so they can verify claims without opening the documents.
3. **Cost cap** (12). Stop researching if `usage['research']` exceeds a budget,
   and write with what you have, flagged.
4. **Time travel** (08). Fork the thread from the checkpoint before `write`,
   change one finding, and see how the briefing changes, without re-running research.
5. **Tests** (12). Build the graph with `FakeMessagesListChatModel` and test that
   a failing review loops at most `MAX_REVISIONS` times, then goes to approval flagged.
6. **Multi-agent** (06). Replace the single writer with a writer + a separate
   "customer-facing tone" editor subgraph (10).

## 10 · Check yourself

<details><summary><b>1.</b> Why does the crash in <code>write</code> not repeat planning or research on resume?</summary>

Each superstep was checkpointed with `durability='sync'` before the next began. Resuming with `None` loads the latest checkpoint, whose next node is `write`, with all findings already in the state.
</details>

<details><summary><b>2.</b> Why is saving a separate node after <code>approval</code>, not part of it?</summary>

On resume the approval node re-runs from the top, so a side effect inside it before `interrupt()` would repeat. And the save must only happen after a human decision. A separate node after the decision runs exactly when it should.
</details>

<details><summary><b>3.</b> Why do the research workers receive the request as well as their question?</summary>

Context. A worker with only a bare question can drift off-topic (notebook 05). The request tells it what the research is for, while the payload stays small.
</details>

<details><summary><b>4.</b> What guarantees the writer ⇄ reviewer loop ends, and what happens to a draft that never passes?</summary>

The `revision` counter checked by `after_review` (`MAX_REVISIONS`). A draft that still fails goes to the human flagged with the reviewer's problems, never silently published.
</details>

<details><summary><b>5.</b> Which parts of this design would you keep if you rebuilt it in another framework?</summary>

All the decisions: plan then parallel research, least-privilege tools, exact checks in code plus model judgement, a capped loop, human approval before the side effect, durable checkpoints. Only the API for expressing them changes, which is what the other sections of Phase 5 show.
</details>

## Section complete 🎉

You've built a durable, parallel, self-checking, human-approved research
workflow on LangGraph, and every design decision in it came from one of
notebooks 01–12.

**Where to go next:** open another framework section of Phase 5 and build the
same project there. The comparison section puts all the versions side by side.